# Backend-10: Remark(PD/Non-PD) 변경 시각 산출 (single-run notebook)

- 대상 테이블: `a1_fct_vision_testlog_txt_processing_history.fct_vision_testlog_txt_processing_history`
- 대상 컬럼: `end_day`(text, YYYYMMDD), `end_time`(text, HH:MI:SS), `station`(Vision1/Vision2), `remark`(PD/Non-PD)
- 출력: `prod_day='20260130'` 고정, `day`와 `night` 각각에 대해 **station별 remark 전환 이벤트**를 산출하여 DataFrame 출력
- 전환 이벤트 정의:
  - 시간 흐름(오름차순) 기준 인접 row에서 `remark`가 바뀌는 지점
  - 이벤트 `at_time` = **바뀌기 전(직전 row)의 `end_time`**
  - `remark`가 없으면(Null/빈값 등) 해당 row는 제외

출력 컬럼(고정):
`prod_day, station, shift_type, at_time, from_remark, to_remark, updated_at`


In [4]:
from __future__ import annotations

from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# =========================
# 0) 고정 파라미터
# =========================
PROD_DAY = "20260128"  # YYYYMMDD (고정)

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SRC_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
SRC_TABLE  = "fct_vision_testlog_txt_processing_history"

STATIONS = ("Vision1", "Vision2")
REMARKS  = ("PD", "Non-PD")

# =========================
# 1) 유틸
# =========================
def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

def prod_day_to_date(prod_day: str) -> date:
    return datetime.strptime(prod_day, "%Y%m%d").date()

def shift_window(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    d = prod_day_to_date(prod_day)
    if shift_type == "day":
        start = datetime.combine(d, time(8, 30, 0))
        end   = datetime.combine(d, time(20, 29, 59))
    elif shift_type == "night":
        start = datetime.combine(d, time(20, 30, 0))
        end   = datetime.combine(d + timedelta(days=1), time(8, 29, 59))
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end

def src_fqn() -> str:
    return f'"{SRC_SCHEMA}"."{SRC_TABLE}"'

def fetch_shift_rows(engine: Engine, shift_type: str) -> pd.DataFrame:
    start_dt, end_dt = shift_window(PROD_DAY, shift_type)

    sql = text(f"""
        SELECT
            station,
            remark,
            end_day,
            end_time,
            to_timestamp(end_day || ' ' || end_time, 'YYYYMMDD HH24:MI:SS')::timestamp AS end_ts
        FROM {src_fqn()}
        WHERE station = ANY(:stations)
          AND remark  = ANY(:remarks)
          AND to_timestamp(end_day || ' ' || end_time, 'YYYYMMDD HH24:MI:SS')::timestamp
              BETWEEN :start_dt AND :end_dt
        ORDER BY station, end_ts
    """)

    with engine.begin() as conn:
        df = pd.read_sql(
            sql,
            conn,
            params={
                "stations": list(STATIONS),
                "remarks": list(REMARKS),
                "start_dt": start_dt,
                "end_dt": end_dt,
            },
        )

    # 안전장치: 혹시라도 Null/빈값이 섞여오면 제외
    df = df.dropna(subset=["station", "remark", "end_day", "end_time", "end_ts"]).copy()
    df["remark"] = df["remark"].astype(str)
    df = df[df["remark"].isin(REMARKS)].copy()

    return df

def build_transition_events(df: pd.DataFrame, shift_type: str) -> pd.DataFrame:
    # station별 시간 오름차순
    df = df.sort_values(["station", "end_ts"], ascending=True).reset_index(drop=True)

    updated_at = pd.Timestamp.now(tz=KST)

    rows = []
    for station, g in df.groupby("station", sort=False):
        g = g.reset_index(drop=True)
        if len(g) < 2:
            continue

        prev_remark = g.loc[0, "remark"]
        prev_end_time = g.loc[0, "end_time"]

        for i in range(1, len(g)):
            cur_remark = g.loc[i, "remark"]
            if cur_remark != prev_remark:
                rows.append(
                    {
                        "prod_day": PROD_DAY,
                        "station": station,
                        "shift_type": shift_type,
                        "at_time": prev_end_time,   # 직전 row end_time (확정)
                        "from_remark": prev_remark,
                        "to_remark": cur_remark,
                        "updated_at": updated_at,
                    }
                )

            prev_remark = cur_remark
            prev_end_time = g.loc[i, "end_time"]

    return pd.DataFrame(
        rows,
        columns=[
            "prod_day",
            "station",
            "shift_type",
            "at_time",
            "from_remark",
            "to_remark",
            "updated_at",
        ],
    )


In [5]:
# =========================
# 2) 실행: day / night
# =========================
engine = make_engine()

df_day_src = fetch_shift_rows(engine, "day")
df_night_src = fetch_shift_rows(engine, "night")

df_day_events = build_transition_events(df_day_src, "day")
df_night_events = build_transition_events(df_night_src, "night")

print(f"[SRC] day rows   = {len(df_day_src):,} / events = {len(df_day_events):,}")
print(f"[SRC] night rows = {len(df_night_src):,} / events = {len(df_night_events):,}")

df_day_events


[SRC] day rows   = 3,221 / events = 2
[SRC] night rows = 3,972 / events = 0


,prod_day,station,shift_type,at_time,from_remark,to_remark,updated_at
0,20260128,Vision1,day,10:56:20,Non-PD,PD,2026-02-02 12:15:26.816824+09:00
1,20260128,Vision2,day,10:58:21,Non-PD,PD,2026-02-02 12:15:26.816824+09:00


In [6]:
df_night_events

,prod_day,station,shift_type,at_time,from_remark,to_remark,updated_at


In [7]:
# =========================
# 3) 저장: i_daily_report.*
# =========================
from sqlalchemy import text

SAVE_SCHEMA = "i_daily_report"
T_DAY   = "j_remark_change_day_daily"
T_NIGHT = "j_remark_change_night_daily"

COLS = [
    "prod_day",
    "station",
    "shift_type",
    "at_time",
    "from_remark",
    "to_remark",
    "updated_at",
]

UNIQUE_COLS = ["prod_day", "station", "shift_type", "at_time", "from_remark", "to_remark"]

def ensure_schema(engine, schema: str):
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}";'))

def ensure_table(engine, schema: str, table: str):
    fqn = f'"{schema}"."{table}"'
    cols_ddl = """
        "prod_day"    text NOT NULL,
        "station"     text NOT NULL,
        "shift_type"  text NOT NULL,
        "at_time"     text NOT NULL,
        "from_remark" text NOT NULL,
        "to_remark"   text NOT NULL,
        "updated_at"  timestamptz NOT NULL
    """
    unique_name = f'ux_{schema}_{table}_dedup'
    unique_cols_sql = ", ".join([f'"{c}"' for c in UNIQUE_COLS])

    with engine.begin() as conn:
        conn.execute(text(f"CREATE TABLE IF NOT EXISTS {fqn} ({cols_ddl});"))
        conn.execute(text(f'CREATE UNIQUE INDEX IF NOT EXISTS "{unique_name}" ON {fqn} ({unique_cols_sql});'))

def upsert_events(engine, schema: str, table: str, df: pd.DataFrame):
    if df is None or df.empty:
        print(f"[SAVE] {schema}.{table}: df is empty -> skip")
        return

    # 컬럼 정렬/누락 방지
    df2 = df.loc[:, COLS].copy()

    fqn = f'"{schema}"."{table}"'
    cols_sql = ", ".join([f'"{c}"' for c in COLS])
    values_sql = ", ".join([f":{c}" for c in COLS])
    conflict_sql = ", ".join([f'"{c}"' for c in UNIQUE_COLS])

    ins = text(f"""
        INSERT INTO {fqn} ({cols_sql})
        VALUES ({values_sql})
        ON CONFLICT ({conflict_sql}) DO NOTHING
    """)

    payload = df2.to_dict("records")
    with engine.begin() as conn:
        conn.execute(ins, payload)

    print(f"[SAVE] {schema}.{table}: inserted (deduped) {len(payload):,} rows")

# 1) 스키마/테이블 보장
ensure_schema(engine, SAVE_SCHEMA)
ensure_table(engine, SAVE_SCHEMA, T_DAY)
ensure_table(engine, SAVE_SCHEMA, T_NIGHT)

# 2) 저장 실행
upsert_events(engine, SAVE_SCHEMA, T_DAY, df_day_events)
upsert_events(engine, SAVE_SCHEMA, T_NIGHT, df_night_events)

[SAVE] i_daily_report.j_remark_change_day_daily: inserted (deduped) 2 rows
[SAVE] i_daily_report.j_remark_change_night_daily: df is empty -> skip
